# KLUE-RoBERTa — test_final 최종 평가

> ⚠️ **이 노트북은 딱 1회만 실행한다. test_final은 최종 평가 전까지 절대 열지 않는다.**

| 항목 | 값 |
|------|----|)
| 모델 | `klue/roberta-base` (전체 데이터 재학습본) |
| 평가 데이터 | `test_final.parquet` (36,434건) |
| 평가 방식 | 추론 전용 (학습 없음) |

## Step 0. 패키지 설치

In [ ]:
!pip install -q transformers torch sentencepiece protobuf scikit-learn tqdm

## Step 1. Google Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/MyDrive/text-mining-2026/data/processed'
SAVE_DIR = '/content/drive/MyDrive/text-mining-2026/models'

print(f'데이터 경로: {DATA_DIR}')
print(f'모델 경로:   {SAVE_DIR}')

## Step 2. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 3. test_final 로딩

In [ ]:
df_test = pd.read_parquet(f'{DATA_DIR}/test_final.parquet')
print(f'test_final: {len(df_test):,}건')
print(f'컬럼: {list(df_test.columns)}')
print(f'\n이진 분류 분포:')
print(df_test['binary_label'].value_counts().sort_index())
print(f'\n다중 분류 분포 (type_label >= 0):')
print(df_test[df_test['type_label'] != -1]['type_label'].value_counts().sort_index())

## Step 4. 설정값

In [ ]:
MODEL_NAME = 'klue/roberta-base'
MODEL_KEY  = 'klue'
MAX_LENGTH = 512
BATCH_SIZE = 64

TYPE_NAMES = {
    0: '의문유발-부호', 1: '의문유발-은닉', 2: '선정표현',
    3: '속어/줄임말',  4: '사실과대',      5: '주어왜곡',
}

# 5-Fold CV 결과 (비교용)
CV_RESULTS = {
    'binary': {'f1_macro': 0.9869, 'accuracy': 0.9869},
    'multi':  {'f1_macro': 0.8854, 'accuracy': 0.9228},
}

print(f'모델: {MODEL_NAME}')

## Step 5. Dataset / 추론 함수

In [ ]:
class ClickbaitDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH, task='binary'):
        if task == 'multi':
            dataframe = dataframe[dataframe['type_label'] != -1].reset_index(drop=True)
        self.titles   = dataframe['title_clean'].tolist()
        self.contents = dataframe['content_clean'].tolist()
        self.labels   = dataframe[
            'binary_label' if task == 'binary' else 'type_label'
        ].tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            text=self.titles[idx],
            text_pair=self.contents[idx],
            truncation='only_second',
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
        }
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def run_inference(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  추론', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels']
            with torch.autocast('cuda', dtype=torch.bfloat16):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


def compute_metrics(labels, preds):
    return {
        'accuracy':        accuracy_score(labels, preds),
        'f1_macro':        f1_score(labels, preds, average='macro', zero_division=0),
        'precision_macro': precision_score(labels, preds, average='macro', zero_division=0),
        'recall_macro':    recall_score(labels, preds, average='macro', zero_division=0),
    }

print('Dataset / 추론 함수 정의 완료')

## Step 6. 이진 분류 최종 평가

In [ ]:
print('=== 이진 분류 최종 평가 ===')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset_bin = ClickbaitDataset(df_test, tok, task='binary')
loader_bin  = DataLoader(dataset_bin, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f'테스트 샘플: {len(dataset_bin):,}건')

model_bin = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_bin.load_state_dict(torch.load(f'{SAVE_DIR}/{MODEL_KEY}_binary_final.pt',
                                     map_location=device))
model_bin.to(device)

labels_bin, preds_bin = run_inference(model_bin, loader_bin)
metrics_bin = compute_metrics(labels_bin, preds_bin)

print(f'\n[이진 분류 결과]')
for k, v in metrics_bin.items():
    cv_val = CV_RESULTS['binary'].get(k)
    diff   = f'  (CV: {cv_val:.4f}, 차이: {v - cv_val:+.4f})' if cv_val else ''
    print(f'  {k:<22}: {v:.4f}{diff}')

del model_bin
torch.cuda.empty_cache()

## Step 7. 다중 분류 최종 평가

In [ ]:
print('=== 다중 분류 최종 평가 ===')

dataset_multi = ClickbaitDataset(df_test, tok, task='multi')
loader_multi  = DataLoader(dataset_multi, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)
print(f'테스트 샘플: {len(dataset_multi):,}건')

model_multi = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=6)
model_multi.load_state_dict(torch.load(f'{SAVE_DIR}/{MODEL_KEY}_multi_final.pt',
                                       map_location=device))
model_multi.to(device)

labels_multi, preds_multi = run_inference(model_multi, loader_multi)
metrics_multi = compute_metrics(labels_multi, preds_multi)

print(f'\n[다중 분류 결과]')
for k, v in metrics_multi.items():
    cv_val = CV_RESULTS['multi'].get(k)
    diff   = f'  (CV: {cv_val:.4f}, 차이: {v - cv_val:+.4f})' if cv_val else ''
    print(f'  {k:<22}: {v:.4f}{diff}')

print(f'\n[Confusion Matrix]')
cm = confusion_matrix(labels_multi, preds_multi)
header = ''.join([f'{TYPE_NAMES[i][:5]:>10}' for i in range(6)])
print(f'  {"":>14}{header}')
for i, row in enumerate(cm):
    print(f'  {TYPE_NAMES[i][:12]:<14}' + ''.join([f'{v:>10,}' for v in row]))

print(f'\n[Classification Report]')
print(classification_report(
    labels_multi, preds_multi,
    target_names=list(TYPE_NAMES.values()), zero_division=0,
))

del model_multi
torch.cuda.empty_cache()

## Step 8. 최종 결과 요약

In [ ]:
print('=' * 60)
print('  KLUE-RoBERTa 최종 평가 요약')
print('=' * 60)
print(f'  {"":22}  {"5-Fold CV":>10}  {"test_final":>10}  {"차이":>8}')
print(f'  {"-"*56}')
for task, label in [('binary', '이진 분류'), ('multi', '다중 분류')]:
    metrics = metrics_bin if task == 'binary' else metrics_multi
    cv_f1   = CV_RESULTS[task]['f1_macro']
    test_f1 = metrics['f1_macro']
    diff    = test_f1 - cv_f1
    print(f'  {label} F1-macro     {cv_f1:>10.4f}  {test_f1:>10.4f}  {diff:>+8.4f}')
print('=' * 60)
if abs(metrics_bin['f1_macro'] - CV_RESULTS['binary']['f1_macro']) < 0.01 and \
   abs(metrics_multi['f1_macro'] - CV_RESULTS['multi']['f1_macro']) < 0.01:
    print('  ✅ CV와 test 결과 일치 — 과적합 없음')
else:
    print('  ⚠️  CV와 test 결과 차이 큼 — 과적합 또는 분포 차이 확인 필요')